# Taller 2 - EcoMarket: asistente de atención al cliente con RAG

En el Taller 1 el asistente de EcoMarket respondía dos tipos de consulta (estado de un pedido y devolución de un producto) porque los datos necesarios se pegaban a mano dentro del prompt. Este notebook extiende ese asistente con un sistema **RAG** (generación aumentada por recuperación): ahora el asistente busca por sí mismo, en una base de conocimiento de la empresa, los fragmentos que necesita para responder cualquier pregunta, y reconoce cuándo no tiene información para hacerlo.

**Componentes elegidos** (la justificación completa está en el `README.md`, Fases 1 y 2):

| Pieza | Elección |
|---|---|
| Modelo de embeddings | `intfloat/multilingual-e5-small` (código abierto, multilingüe, se ejecuta en local) |
| Base de datos vectorial | ChromaDB persistente en disco |
| Framework de orquestación | LangChain (LCEL) |
| Modelo de lenguaje | Gemini por API, usando el mismo cliente compatible con OpenAI del Taller 1 |

**Flujo del sistema**

```
INDEXACIÓN (una vez)
Documentos (PDF, CSV, JSON) -> carga -> segmentación -> embeddings -> ChromaDB

CONSULTA (cada pregunta)
Pregunta -> embedding de la pregunta -> búsqueda por similitud en ChromaDB
         -> fragmentos más parecidos -> prompt (reglas + contexto + pregunta)
         -> modelo de lenguaje -> respuesta con sus fuentes
```

**Contenido**
1. Configuración
2. Carga de los documentos
3. Segmentación (chunking)
4. Modelo de embeddings
5. Indexación en la base de datos vectorial
6. Recuperación
7. Experimento 1: efecto de la estrategia de segmentación
8. Modelo de lenguaje
9. Prompt y cadena RAG
10. Pruebas del asistente
11. Experimentos 2 a 5: cambios en el prompt y en la recuperación
12. Evaluación
13. Limitaciones y suposiciones
14. Conclusiones
15. Anexo: cómo se generó el PDF de la política

## 1. Configuración

Todos los parámetros del sistema están en esta celda. La clave del modelo de lenguaje no se escribe en el notebook: se lee del archivo `.env` (ver `.env.example`), que está excluido del repositorio.

In [1]:
import json
import os
import re
import time
import unicodedata
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import pymupdf
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)


def ubicar_raiz():
    """Busca hacia arriba la carpeta del proyecto (la que contiene 'Datos')."""
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "Datos").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la carpeta 'Datos' del proyecto.")


RUTA_BASE = ubicar_raiz()
RUTA_DATOS = RUTA_BASE / "Datos"
RUTA_SALIDA = RUTA_BASE / "Salida"
RUTA_CHROMA = RUTA_BASE / "chroma_db"
RUTA_SALIDA.mkdir(exist_ok=True)

# El archivo .env puede estar junto al notebook o en la raíz del proyecto
for carpeta in [Path.cwd(), RUTA_BASE]:
    if (carpeta / ".env").exists():
        load_dotenv(carpeta / ".env")
        break

# Recuperación
MODELO_EMBEDDINGS = os.getenv("MODELO_EMBEDDINGS", "intfloat/multilingual-e5-small")
NOMBRE_COLECCION = "ecomarket"
TAMANO_CHUNK = 500        # caracteres
SOLAPAMIENTO = 80         # caracteres
TOP_K = 4                 # fragmentos que se entregan al modelo
UMBRAL_SIMILITUD = 0.84   # se calibra en la sección 6

# Generación
MODELO_LLM = os.getenv("MODELO_LLM", "gemini-3.5-flash-lite")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
PAUSA_ENTRE_LLAMADAS = float(os.getenv("PAUSA_ENTRE_LLAMADAS", "4"))  # segundos, por el límite del plan gratuito

print("Carpeta del proyecto:", RUTA_BASE.name)
print("Documentos disponibles:", sorted(p.name for p in RUTA_DATOS.iterdir() if p.is_file()))
print("Modelo de embeddings:", MODELO_EMBEDDINGS)
print("Modelo de lenguaje:", MODELO_LLM)
print("Clave del modelo de lenguaje configurada:", bool(LLM_API_KEY))

Carpeta del proyecto: Taller_2
Documentos disponibles: ['inventario_productos.csv', 'pedidos.csv', 'politica_devoluciones.pdf', 'preguntas_frecuentes.json']
Modelo de embeddings: intfloat/multilingual-e5-small
Modelo de lenguaje: gemini-3.5-flash-lite
Clave del modelo de lenguaje configurada: True


## 2. Carga de los documentos

La base de conocimiento tiene cuatro archivos de tres formatos distintos. Cada formato necesita su propio cargador, porque la unidad de información es diferente en cada uno:

| Archivo | Formato | Unidad de información | Origen |
|---|---|---|---|
| `politica_devoluciones.pdf` | PDF | Sección de la política | Sintético, con las reglas de devolución del Taller 1 |
| `inventario_productos.csv` | CSV | Un producto (una fila) | Sintético |
| `pedidos.csv` | CSV | Un pedido (una fila) | Los 10 pedidos reales de Olist usados en el Taller 1 |
| `preguntas_frecuentes.json` | JSON | Un par pregunta-respuesta | Sintético |

Cada unidad se convierte en un objeto `Document` de LangChain: un texto (`page_content`) más unos metadatos (`metadata`) que guardan de dónde salió. Los metadatos sirven después para citar la fuente y para filtrar.

Las filas de los CSV se reescriben como frases ("Producto ECO-007: Botella térmica... Precio: 69900 pesos"). Un modelo de embeddings entiende lenguaje natural, no celdas separadas por comas, y una fila suelta pierde los nombres de sus columnas.

In [2]:
from langchain_core.documents import Document


def cargar_politica_pdf(ruta):
    """Lee el PDF y devuelve un Document por sección, conservando sus párrafos."""
    pdf = pymupdf.open(ruta)
    bloques = []
    for pagina in pdf:
        # "blocks" entrega el texto agrupado por párrafo; get_text() simple lo entrega por renglón
        for bloque in pagina.get_text("blocks"):
            if bloque[6] == 0:  # 0 = bloque de texto, 1 = imagen
                # NFKC deshace las ligaduras tipográficas del PDF ("ﬁ" vuelve a ser "fi")
                texto = unicodedata.normalize("NFKC", " ".join(bloque[4].split()))
                if texto:
                    bloques.append(texto)
    pdf.close()

    titulo = bloques[0]
    secciones = []
    for bloque in bloques[1:]:
        es_titulo_de_seccion = re.match(r"^\d+\.\s", bloque) and len(bloque) < 80
        if es_titulo_de_seccion:
            secciones.append({"seccion": bloque, "parrafos": []})
        elif secciones:
            secciones[-1]["parrafos"].append(bloque)

    return [
        Document(
            page_content="\n\n".join(s["parrafos"]),
            metadata={"fuente": ruta.name, "tipo": "politica", "titulo": titulo, "seccion": s["seccion"]},
        )
        for s in secciones
    ]


def cargar_inventario_csv(ruta):
    """Un Document por producto, redactado como frase."""
    documentos = []
    for _, fila in pd.read_csv(ruta).iterrows():
        disponibilidad = f"{fila['stock']} unidades disponibles" if fila["stock"] > 0 else "agotado, 0 unidades disponibles"
        garantia = f"{fila['garantia_meses']} meses" if fila["garantia_meses"] > 0 else "no tiene garantía extendida"
        texto = (
            f"Producto {fila['id_producto']}: {fila['nombre']}. Categoría: {fila['categoria']}. "
            f"Descripción: {fila['descripcion']}. Material: {fila['material']}. "
            f"Precio: {fila['precio_cop']} pesos colombianos. Inventario: {disponibilidad}. "
            f"Garantía: {garantia}. Certificación: {fila['certificacion']}."
        )
        documentos.append(Document(
            page_content=texto,
            metadata={"fuente": ruta.name, "tipo": "producto", "id_producto": fila["id_producto"]},
        ))
    return documentos


def cargar_pedidos_csv(ruta):
    """Un Document por pedido. El número de pedido queda en los metadatos para poder filtrar por él."""
    documentos = []
    for _, fila in pd.read_csv(ruta).iterrows():
        def valor(columna, vacio="no registrado"):
            return vacio if pd.isna(fila[columna]) else fila[columna]

        texto = (
            f"Pedido número {fila['numero_pedido']}. Estado: {fila['estado']}. "
            f"Fecha de compra: {valor('fecha_compra')}. "
            f"Fecha estimada de entrega: {valor('fecha_entrega_estimada')}. "
            f"Fecha de entrega real: {valor('fecha_entrega_real', 'no registrada, el pedido no ha sido entregado')}. "
            f"Categoría del producto: {valor('categoria_producto', 'no registrada')}. "
            f"Precio del producto: {valor('precio_producto')}. Costo de transporte: {valor('costo_transporte')}."
        )
        documentos.append(Document(
            page_content=texto,
            metadata={"fuente": ruta.name, "tipo": "pedido", "numero_pedido": fila["numero_pedido"]},
        ))
    return documentos


def cargar_faq_json(ruta):
    """Un Document por pregunta frecuente: la pregunta y su respuesta van juntas."""
    preguntas = json.loads(ruta.read_text(encoding="utf-8"))
    return [
        Document(
            page_content=f"Pregunta frecuente ({p['categoria']}): {p['pregunta']}\nRespuesta: {p['respuesta']}",
            metadata={"fuente": ruta.name, "tipo": "faq", "id_faq": p["id"], "categoria": p["categoria"]},
        )
        for p in preguntas
    ]


documentos = (
    cargar_politica_pdf(RUTA_DATOS / "politica_devoluciones.pdf")
    + cargar_inventario_csv(RUTA_DATOS / "inventario_productos.csv")
    + cargar_pedidos_csv(RUTA_DATOS / "pedidos.csv")
    + cargar_faq_json(RUTA_DATOS / "preguntas_frecuentes.json")
)

resumen_carga = pd.DataFrame(
    [{"fuente": d.metadata["fuente"], "tipo": d.metadata["tipo"], "caracteres": len(d.page_content)} for d in documentos]
)
display(resumen_carga.groupby(["fuente", "tipo"])["caracteres"].agg(unidades="count", minimo="min", promedio="mean", maximo="max").round(0))

print("\nEjemplo de cada tipo:")
for tipo in ["politica", "producto", "pedido", "faq"]:
    ejemplo = next(d for d in documentos if d.metadata["tipo"] == tipo)
    print(f"\n[{tipo}] {ejemplo.page_content[:260]}")

,,unidades,minimo,promedio,maximo
fuente,tipo,,,,
inventario_productos.csv,producto,16,304,334.0,369
pedidos.csv,pedido,10,261,293.0,319
politica_devoluciones.pdf,politica,11,184,342.0,595
preguntas_frecuentes.json,faq,15,134,255.0,496



Ejemplo de cada tipo:

[politica] Esta política describe las condiciones bajo las cuales los clientes de EcoMarket pueden devolver un producto y recibir un reembolso o un cambio. Aplica a todas las compras realizadas en la tienda en línea de EcoMarket con entrega dentro de Colombia.

[producto] Producto ECO-001: Cepillo de dientes de bambú. Categoría: Belleza y salud. Descripción: Cepillo dental con mango de bambú y cerdas suaves, paquete de 4 unidades. Material: Bambú moso y nailon libre de BPA. Precio: 18900 pesos colombianos. Inventario: 240 unida

[pedido] Pedido número e481f51cbdc54678b7cc49136f2d6af7. Estado: Entregado. Fecha de compra: 2017-10-02. Fecha estimada de entrega: 2017-10-18. Fecha de entrega real: 2017-10-10. Categoría del producto: Utilidades domésticas. Precio del producto: 29.99. Costo de transp

[faq] Pregunta frecuente (Envíos): ¿Cuánto tarda en llegar mi pedido?
Respuesta: Los pedidos llegan en 3 a 5 días hábiles a ciudades principales y en 5 a 8 días hábiles a

## 3. Segmentación (chunking)

Segmentar es decidir qué trozo de texto recibe un vector. Importa por dos razones opuestas:

- Un fragmento **demasiado grande** mezcla varios temas: su vector es un promedio que no se parece a ninguna pregunta concreta, y además gasta el espacio del prompt con texto que no sirve.
- Un fragmento **demasiado pequeño** o cortado en mal sitio deja la respuesta partida: la regla en un fragmento y su excepción en otro.

La estrategia depende del tipo de documento:

- **CSV y JSON: un registro por fragmento.** Un producto, un pedido o una pregunta frecuente ya son unidades completas y cortas. Dividirlas separaría el nombre del producto de su precio; unirlas mezclaría productos.
- **PDF: división recursiva dentro de cada sección.** Primero se respeta la estructura del documento (una sección por unidad). Solo las secciones que superan `TAMANO_CHUNK` se dividen, y la división recursiva intenta cortar primero por párrafo, después por frase y solo al final por palabra. Así un corte casi nunca cae en mitad de una frase.
- **Título de la sección en cada fragmento.** A cada fragmento del PDF se le antepone el título del documento y de su sección. Sin eso, un fragmento como "Los perfumes que hayan sido abiertos o utilizados no pueden ser devueltos" quedaría bien, pero "Tampoco se aceptan..." perdería el tema del que habla.

El solapamiento (`SOLAPAMIENTO`) repite el final de un fragmento al inicio del siguiente cuando el corte cae entre frases de un mismo párrafo, para que una idea que cruza el corte no se pierda.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def segmentar(documentos, tamano=TAMANO_CHUNK, solapamiento=SOLAPAMIENTO):
    """Aplica a cada documento la estrategia de segmentación que corresponde a su tipo."""
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=tamano,
        chunk_overlap=solapamiento,
        separators=["\n\n", ". ", " ", ""],  # párrafo, frase, palabra, carácter
        keep_separator="end",
    )
    fragmentos = []
    for documento in documentos:
        if documento.metadata["tipo"] != "politica":
            fragmentos.append(documento)  # un registro ya es un fragmento completo
            continue
        encabezado = f"{documento.metadata['titulo']} - {documento.metadata['seccion']}"
        for numero, parte in enumerate(divisor.split_text(documento.page_content), start=1):
            fragmentos.append(Document(
                page_content=f"{encabezado}\n{parte.strip()}",
                metadata={**documento.metadata, "parte": numero},
            ))
    return fragmentos


fragmentos = segmentar(documentos)

largos = pd.DataFrame([{"tipo": f.metadata["tipo"], "caracteres": len(f.page_content)} for f in fragmentos])
print(f"Unidades cargadas: {len(documentos)}  ->  fragmentos a indexar: {len(fragmentos)}")
display(largos.groupby("tipo")["caracteres"].agg(fragmentos="count", minimo="min", promedio="mean", maximo="max").round(0))

seccion_dividida = [f for f in fragmentos if f.metadata.get("seccion", "").startswith("7.")]
print(f"\nLa sección 7 de la política no cabe en {TAMANO_CHUNK} caracteres y queda en {len(seccion_dividida)} fragmentos:")
for f in seccion_dividida:
    print(f"\n--- parte {f.metadata['parte']} ({len(f.page_content)} caracteres) ---\n{f.page_content}")

Unidades cargadas: 52  ->  fragmentos a indexar: 54


,fragmentos,minimo,promedio,maximo
tipo,,,,
faq,15,134,255.0,496
pedido,10,261,293.0,319
politica,13,199,372.0,566
producto,16,304,334.0,369



La sección 7 de la política no cabe en 500 caracteres y queda en 2 fragmentos:

--- parte 1 (566 caracteres) ---
Política de Devoluciones y Reembolsos de EcoMarket - 7. Cómo solicitar una devolución
Paso 1: ingresar a la sección Mis pedidos de la cuenta, seleccionar el pedido y elegir la opción Solicitar devolución, o escribir a devoluciones@ecomarket.example indicando el número de pedido y el motivo.

Paso 2: EcoMarket responde la solicitud en un máximo de 2 días hábiles y, si es aprobada, envía una guía de transporte al correo del cliente.

Paso 3: el cliente empaca el producto y lo entrega a la transportadora dentro de los 7 días calendario siguientes a la aprobación.

--- parte 2 (199 caracteres) ---
Política de Devoluciones y Reembolsos de EcoMarket - 7. Cómo solicitar una devolución
Paso 4: EcoMarket revisa el producto recibido en un máximo de 3 días hábiles y confirma el reembolso o el cambio.


## 4. Modelo de embeddings

Un modelo de embeddings convierte un texto en un vector de números de modo que dos textos con significado parecido quedan cerca en ese espacio, aunque no compartan palabras. Esa es la pieza que permite que "¿me devuelven la plata?" encuentre el fragmento que habla de "reembolso".

Se usa `intfloat/multilingual-e5-small`: 384 dimensiones, entrenado en cerca de 100 idiomas incluido el español, y lo bastante liviano para ejecutarse en CPU. El ejemplo de clase usaba `all-MiniLM-L6-v2`, que fue entrenado sobre todo con texto en inglés.

Los modelos E5 exigen un prefijo que les indica el papel del texto: `query: ` para la pregunta y `passage: ` para los fragmentos. LangChain no conoce esa regla, así que se escribe una clase pequeña que cumple la interfaz `Embeddings` de LangChain y agrega el prefijo correcto en cada caso. Los vectores se normalizan a longitud 1, de modo que la similitud coseno es simplemente el producto punto.

In [4]:
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer


class EmbeddingsE5(Embeddings):
    """Adapta un modelo E5 de sentence-transformers a la interfaz de LangChain."""

    def __init__(self, nombre_modelo):
        self.modelo = SentenceTransformer(nombre_modelo, device="cpu")

    def embed_documents(self, textos):
        vectores = self.modelo.encode([f"passage: {t}" for t in textos], normalize_embeddings=True)
        return vectores.tolist()

    def embed_query(self, texto):
        return self.modelo.encode(f"query: {texto}", normalize_embeddings=True).tolist()


embeddings = EmbeddingsE5(MODELO_EMBEDDINGS)

pregunta_demo = "¿Me devuelven la plata si el producto llega roto?"
frases_demo = [
    "En productos defectuosos o dañados el cliente puede elegir el reembolso total.",
    "El envío estándar cuesta 9.900 pesos a ciudades principales.",
    "La lámpara de mesa está hecha de fique y hierro reciclado.",
]
vector_pregunta = np.array(embeddings.embed_query(pregunta_demo))
vectores_frases = np.array(embeddings.embed_documents(frases_demo))

print("Dimensiones de cada vector:", vector_pregunta.shape[0])
print(f"\nPregunta: {pregunta_demo}")
for frase, similitud in zip(frases_demo, vectores_frases @ vector_pregunta):
    print(f"  similitud {similitud:.3f}  |  {frase}")

Dimensiones de cada vector: 384

Pregunta: ¿Me devuelven la plata si el producto llega roto?
  similitud 0.847  |  En productos defectuosos o dañados el cliente puede elegir el reembolso total.
  similitud 0.783  |  El envío estándar cuesta 9.900 pesos a ciudades principales.
  similitud 0.783  |  La lámpara de mesa está hecha de fique y hierro reciclado.


La frase sobre reembolso por daño obtiene la similitud más alta aunque no comparte con la pregunta las palabras "plata" ni "roto". También se ve una característica de E5 que hay que tener en cuenta más adelante: las similitudes viven en un rango estrecho y alto, y textos sin relación no bajan a cero. Por eso el umbral de relevancia de la sección 6 se calibra con datos en lugar de fijarse a ojo.

## 5. Indexación en la base de datos vectorial

Indexar es calcular el vector de cada fragmento y guardarlo junto con su texto y sus metadatos. ChromaDB construye sobre esos vectores un índice HNSW, que encuentra los vecinos más cercanos sin comparar la pregunta contra todos los fragmentos.

- `PersistentClient` guarda el índice en la carpeta `chroma_db/`, así que no hay que recalcular los embeddings en cada ejecución. Aquí se borra y se reconstruye la colección para que el notebook dé siempre el mismo resultado.
- `hnsw:space = cosine` hace que Chroma mida distancia coseno. La similitud que se muestra es `1 - distancia`.
- Cada fragmento recibe un identificador estable, lo que permite actualizar un solo registro (por ejemplo, el inventario de un producto) sin reindexar todo.

In [5]:
import chromadb
from langchain_chroma import Chroma

cliente_chroma = chromadb.PersistentClient(path=str(RUTA_CHROMA))
if NOMBRE_COLECCION in [c.name for c in cliente_chroma.list_collections()]:
    cliente_chroma.delete_collection(NOMBRE_COLECCION)

almacen = Chroma(
    client=cliente_chroma,
    collection_name=NOMBRE_COLECCION,
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)


def identificador(fragmento):
    m = fragmento.metadata
    if m["tipo"] == "politica":
        return f"politica-s{m['seccion'].split('.')[0]}-p{m['parte']}"
    return m.get("id_producto") or m.get("numero_pedido") or m.get("id_faq")


inicio = time.time()
almacen.add_documents(fragmentos, ids=[identificador(f) for f in fragmentos])
print(f"Fragmentos indexados: {almacen._collection.count()} en {time.time() - inicio:.1f} s")
print("Índice guardado en:", RUTA_CHROMA.name)

Fragmentos indexados: 54 en 3.6 s
Índice guardado en: chroma_db


## 6. Recuperación

La función `recuperar` recibe la pregunta y devuelve los fragmentos que se le pasarán al modelo. Tiene tres decisiones de diseño:

1. **Búsqueda semántica** para las preguntas sobre políticas, productos y preguntas frecuentes: se calcula el vector de la pregunta y se piden los `k` fragmentos más cercanos.
2. **Filtro exacto por metadatos para los pedidos.** Un número de pedido es un identificador, no un texto con significado: dos números distintos son, para un modelo de embeddings, casi iguales. Si la pregunta trae un número de pedido, ese pedido se busca por coincidencia exacta en los metadatos. Además, los pedidos se excluyen de la búsqueda semántica, para que el asistente nunca muestre el pedido de otro cliente por simple parecido.
3. **Umbral de relevancia.** Los fragmentos con similitud menor a `UMBRAL_SIMILITUD` se descartan. Si no queda ninguno, el sistema responde que no tiene información sin llamar al modelo de lenguaje.

In [6]:
PATRON_PEDIDO = re.compile(r"\b[0-9a-f]{32}\b")  # formato de los números de pedido


def recuperar(pregunta, k=TOP_K, umbral=UMBRAL_SIMILITUD):
    """Devuelve los fragmentos relevantes para la pregunta, con su similitud en los metadatos."""
    encontrados = []

    numero = PATRON_PEDIDO.search(pregunta.lower())
    if numero:
        numero = numero.group()
        exacto = almacen.get(where={"numero_pedido": numero})
        if exacto["documents"]:
            texto, metadatos = exacto["documents"][0], exacto["metadatas"][0]
        else:
            texto = f"No existe ningún pedido registrado con el número {numero}."
            metadatos = {"fuente": "pedidos.csv", "tipo": "pedido"}
        encontrados.append(Document(page_content=texto, metadata={**metadatos, "similitud": 1.0, "via": "filtro exacto"}))

    semanticos = almacen.similarity_search_with_score(
        pregunta,
        k=k - len(encontrados),
        filter={"tipo": {"$ne": "pedido"}},
    )
    for documento, distancia in semanticos:
        similitud = round(1 - distancia, 3)
        if similitud >= umbral:
            encontrados.append(Document(
                page_content=documento.page_content,
                metadata={**documento.metadata, "similitud": similitud, "via": "semántica"},
            ))
    return encontrados


def mostrar_recuperacion(pregunta, **opciones):
    print(f"\nPregunta: {pregunta}")
    resultados = recuperar(pregunta, **opciones)
    if not resultados:
        print("  (ningún fragmento supera el umbral de similitud)")
    for r in resultados:
        etiqueta = r.metadata.get("seccion") or r.metadata.get("id_producto") or r.metadata.get("id_faq") or "pedido"
        cuerpo = r.page_content.split("\n", 1)[-1] if r.metadata["tipo"] in ("politica", "faq") else r.page_content
        print(f"  {r.metadata['similitud']:.3f}  {r.metadata['fuente']:<26} {etiqueta[:38]:<38} | {cuerpo[:68]}")


mostrar_recuperacion("Abrí un perfume y no me gustó, ¿lo puedo devolver?")
mostrar_recuperacion("¿Cuánto vale la botella térmica y cuántas quedan?")
mostrar_recuperacion("¿Cuál es el estado de mi pedido ee64d42b8cf066f35eac1cf57de1aa85?")
mostrar_recuperacion("¿Dónde está mi pedido 00000000000000000000000000000000?")


Pregunta: Abrí un perfume y no me gustó, ¿lo puedo devolver?
  0.874  politica_devoluciones.pdf  4. Productos que no se pueden devolver | Por razones sanitarias, los productos de higiene personal que hayan 
  0.851  politica_devoluciones.pdf  3. Productos que sí se pueden devolver | Los productos que llegaron defectuosos pueden ser devueltos. Los pro
  0.843  inventario_productos.csv   ECO-014                                | Producto ECO-014: Perfume sólido de vainilla. Categoría: Perfumería.
  0.842  inventario_productos.csv   ECO-013                                | Producto ECO-013: Perfume botánico de cítricos. Categoría: Perfumerí

Pregunta: ¿Cuánto vale la botella térmica y cuántas quedan?
  0.904  inventario_productos.csv   ECO-007                                | Producto ECO-007: Botella térmica de acero. Categoría: Utilidades do
  0.860  politica_devoluciones.pdf  9. Garantía                            | Los productos de la categoría Informática y accesorios y las botella



  1.000  pedidos.csv                pedido                                 | No existe ningún pedido registrado con el número 0000000000000000000
  0.868  preguntas_frecuentes.json  FAQ-05                                 | Respuesta: Con el número de pedido que llega al correo de confirmaci
  0.842  preguntas_frecuentes.json  FAQ-06                                 | Respuesta: En procesamiento: el pago fue recibido y el pedido se est


### Por qué los pedidos no se buscan por similitud

La celda siguiente hace a propósito lo que la función `recuperar` evita: buscar un número de pedido por similitud semántica entre los 10 pedidos. Se prueba con los 10 números reales y con un número que no existe.

In [7]:
def buscar_pedido_por_similitud(numero, k=2):
    return almacen.similarity_search_with_score(f"¿Cuál es el estado de mi pedido {numero}?", k=k, filter={"tipo": "pedido"})


filas = []
for pedido in [f for f in fragmentos if f.metadata["tipo"] == "pedido"]:
    numero = pedido.metadata["numero_pedido"]
    (primero, d1), (_, d2) = buscar_pedido_por_similitud(numero)
    filas.append({
        "pedido_buscado": numero,
        "primero_es_el_correcto": primero.metadata["numero_pedido"] == numero,
        "similitud_primero": 1 - d1,
        "similitud_segundo": 1 - d2,
        "margen": d2 - d1,
    })
prueba_pedidos = pd.DataFrame(filas)
display(prueba_pedidos.round(3))
print(f"Pedido correcto en primer lugar: {prueba_pedidos['primero_es_el_correcto'].sum()} de {len(prueba_pedidos)}")
print(f"Margen promedio entre el primero y el segundo: {prueba_pedidos['margen'].mean():.3f}")

numero_inexistente = "11111111111111111111111111111111"
(devuelto, distancia), _ = buscar_pedido_por_similitud(numero_inexistente)
print(f"\nPedido buscado (no existe): {numero_inexistente}")
print(f"La búsqueda semántica devuelve: {devuelto.metadata['numero_pedido']} con similitud {1 - distancia:.3f}")

,pedido_buscado,primero_es_el_correcto,similitud_primero,similitud_segundo,margen
0,e481f51cbdc54678b7cc49136f2d6af7,True,0.864,0.863,0.001
1,53cdb2fc8bc7dce0b6741e2150273451,True,0.875,0.870,0.005
2,47770eb9100c2d0c44946d9cf07ec65d,True,0.883,0.870,0.013
3,ee64d42b8cf066f35eac1cf57de1aa85,True,0.868,0.864,0.004
4,6942b8da583c2f9957e990d028607019,True,0.867,0.859,0.008
5,1b9ecfe83cdc259250e1a8aca174f0ad,True,0.877,0.872,0.006
6,714fb133a6730ab81fa1d3c1b2007291,True,0.872,0.866,0.005
7,8e24261a7e58791d10cb1bf9da94df5c,True,0.883,0.870,0.012
8,136cce7faa42fdb2cefd53fdc79a6098,True,0.870,0.857,0.013
9,15bed8e2fec7fdbadb186b57c46c92f2,True,0.879,0.875,0.004


Pedido correcto en primer lugar: 10 de 10
Margen promedio entre el primero y el segundo: 0.007

Pedido buscado (no existe): 11111111111111111111111111111111
La búsqueda semántica devuelve: 1b9ecfe83cdc259250e1a8aca174f0ad con similitud 0.859


**Lectura del resultado.** En este conjunto la búsqueda semántica sí puso el pedido correcto en primer lugar en los 10 casos, así que el problema no es que falle siempre. El problema es que acierta sin margen y no sabe decir que no:

- La diferencia de similitud entre el pedido correcto y el segundo es de 0,007 en promedio y llega a 0,001. Con 10 pedidos alcanza; con los miles de pedidos de una tienda real, nada garantiza que el correcto siga de primero.
- Para un número que no existe, la búsqueda devuelve de todos modos el pedido de otro cliente, con similitud 0,859. Ese valor queda dentro del rango de los aciertos (0,864 a 0,883), así que un umbral no puede separar los dos casos.

El filtro exacto por metadatos resuelve las dos cosas: devuelve el pedido pedido o ninguno, y permite responder "ese pedido no existe" en lugar de mostrar datos de otra persona.

### Calibración del umbral de relevancia

Para elegir `UMBRAL_SIMILITUD` se compara la similitud del mejor fragmento en dos grupos de preguntas: las que la base de conocimiento sí puede responder y las que no tienen nada que ver con EcoMarket.

In [8]:
PREGUNTAS_CON_RESPUESTA = [
    # (pregunta, textos que debe contener el fragmento que la responde)
    ("¿Cuántos días tengo para devolver un producto?", ["30 días calendario"]),
    ("Abrí un perfume y no me gustó, ¿lo puedo devolver?", ["perfumes que hayan sido abiertos"]),
    ("¿En cuánto tiempo me devuelven la plata de una devolución?", ["entre 5 y 10 días hábiles"]),
    ("Mi producto llegó roto, ¿quién paga el envío de vuelta?", ["asume el costo del envío de retorno"]),
    ("¿Cuáles son los pasos para pedir una devolución?", ["Paso 1"]),
    ("¿Cuánto vale la botella térmica?", ["Botella térmica", "69900"]),
    ("¿Tienen disponible el desodorante natural?", ["Desodorante natural en crema", "Cruelty Free"]),
    ("¿Qué garantía tiene el teclado de bambú?", ["Teclado inalámbrico de bambú", "FSC y RoHS"]),
    ("¿Hacen envíos fuera de Colombia?", ["solo realiza envíos dentro de Colombia"]),
    ("¿Puedo pagar con Nequi?", ["Nequi"]),
    ("¿A qué hora atienden los asesores humanos?", ["8:00 a.m. a 6:00 p.m."]),
    ("¿Cómo compensan las emisiones de carbono de los envíos?", ["bonos de carbono"]),
    ("¿Qué pasa si mi pedido aparece como no disponible?", ["No disponible: el producto se agotó"]),
    ("¿Cuánto cuesta el envío a una ciudad principal?", ["9.900 pesos"]),
]
PREGUNTAS_FUERA_DE_ALCANCE = [
    "¿Cuál es la capital de Francia?",
    "Dame una receta de lasaña",
    "¿Quién ganó el mundial de fútbol de 2014?",
    "¿A cómo está el dólar hoy?",
    "Escríbeme un poema sobre el mar",
    "¿Cómo se resuelve una ecuación de segundo grado?",
]


def mejor_similitud(pregunta):
    _, distancia = almacen.similarity_search_with_score(pregunta, k=1, filter={"tipo": {"$ne": "pedido"}})[0]
    return 1 - distancia


calibracion = pd.DataFrame(
    [{"grupo": "con respuesta en la base", "pregunta": p, "mejor_similitud": mejor_similitud(p)} for p, _ in PREGUNTAS_CON_RESPUESTA]
    + [{"grupo": "fuera de alcance", "pregunta": p, "mejor_similitud": mejor_similitud(p)} for p in PREGUNTAS_FUERA_DE_ALCANCE]
)
display(calibracion.groupby("grupo")["mejor_similitud"].agg(["count", "min", "mean", "max"]).round(3))
display(calibracion.sort_values("mejor_similitud").round(3).reset_index(drop=True))

minimo_con_respuesta = calibracion.loc[calibracion["grupo"] == "con respuesta en la base", "mejor_similitud"].min()
maximo_fuera = calibracion.loc[calibracion["grupo"] == "fuera de alcance", "mejor_similitud"].max()
print(f"Mínimo de las preguntas con respuesta: {minimo_con_respuesta:.3f}")
print(f"Máximo de las preguntas fuera de alcance: {maximo_fuera:.3f}")
print(f"Umbral en uso: {UMBRAL_SIMILITUD}")

,count,min,mean,max
grupo,,,,
con respuesta en la base,14,0.865,0.886,0.915
fuera de alcance,6,0.714,0.777,0.819


,grupo,pregunta,mejor_similitud
0,fuera de alcance,¿Quién ganó el mundial de fútbol de 2014?,0.714
1,fuera de alcance,¿Cuál es la capital de Francia?,0.742
2,fuera de alcance,Escríbeme un poema sobre el mar,0.786
3,fuera de alcance,¿Cómo se resuelve una ecuación de segundo grado?,0.788
4,fuera de alcance,¿A cómo está el dólar hoy?,0.815
5,fuera de alcance,Dame una receta de lasaña,0.819
6,con respuesta en la base,¿Puedo pagar con Nequi?,0.865
7,con respuesta en la base,¿Tienen disponible el desodorante natural?,0.866
8,con respuesta en la base,"Mi producto llegó roto, ¿quién paga el envío de vuelta?",0.874
9,con respuesta en la base,"Abrí un perfume y no me gustó, ¿lo puedo devolver?",0.874


Mínimo de las preguntas con respuesta: 0.865
Máximo de las preguntas fuera de alcance: 0.819
Umbral en uso: 0.84


**Lectura del resultado.** La pregunta con respuesta que peor puntúa obtiene 0,865 y la pregunta ajena que mejor puntúa obtiene 0,819. El umbral se fija en **0,84**, en el espacio entre los dos grupos. Con el valor inicial de 0,80 habrían pasado dos preguntas ajenas (la del dólar y la de la lasaña).

Dos advertencias sobre este umbral:

- El espacio entre los grupos es de solo 0,046, porque E5 comprime todas las similitudes en un rango alto. El valor está calibrado con 20 preguntas y habría que recalibrarlo si cambian el modelo de embeddings o la base de conocimiento.
- El umbral solo detecta preguntas de otro tema. Una pregunta del dominio de EcoMarket cuya respuesta no está en la base (por ejemplo, si venden televisores) se parece mucho a los fragmentos del inventario y supera el umbral. Ese caso lo debe resolver la regla 3 del prompt, y se prueba en el Experimento 3.

## 7. Experimento 1: efecto de la estrategia de segmentación

Este experimento no usa el modelo de lenguaje: mide solo la recuperación. Se indexa la misma base de conocimiento de tres formas y se cuenta en cuántas de las 14 preguntas de la sección anterior el fragmento que contiene la respuesta completa aparece entre los primeros `k` resultados.

- **Tamaño fijo:** el texto crudo de cada archivo se corta cada 300 caracteres, sin mirar el contenido.
- **Recursiva genérica:** el texto crudo se divide con `RecursiveCharacterTextSplitter` (500 caracteres, solapamiento 80), sin conocer la estructura de cada archivo.
- **Por estructura (la elegida):** la de la sección 3.

Los pedidos se dejan fuera porque no se recuperan por similitud.

In [9]:
def texto_crudo(ruta):
    if ruta.suffix == ".pdf":
        pdf = pymupdf.open(ruta)
        texto = "\n".join(pagina.get_text() for pagina in pdf)
        pdf.close()
        return unicodedata.normalize("NFKC", texto)
    return ruta.read_text(encoding="utf-8")


archivos_semanticos = ["politica_devoluciones.pdf", "inventario_productos.csv", "preguntas_frecuentes.json"]
crudos = [texto_crudo(RUTA_DATOS / nombre) for nombre in archivos_semanticos]

divisor_generico = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
estrategias = {
    "Tamaño fijo (300)": [t[i:i + 300] for t in crudos for i in range(0, len(t), 300)],
    "Recursiva genérica (500/80)": [parte for t in crudos for parte in divisor_generico.split_text(t)],
    "Por estructura (elegida)": [f.page_content for f in fragmentos if f.metadata["tipo"] != "pedido"],
}


def tasa_de_acierto(textos, k):
    """Fracción de preguntas cuyo fragmento-respuesta está entre los k más similares."""
    vectores = np.array(embeddings.embed_documents(textos))
    aciertos = 0
    for pregunta, esperados in PREGUNTAS_CON_RESPUESTA:
        similitudes = vectores @ np.array(embeddings.embed_query(pregunta))
        mejores = [textos[i] for i in np.argsort(similitudes)[::-1][:k]]
        aciertos += any(all(e in texto for e in esperados) for texto in mejores)
    return aciertos / len(PREGUNTAS_CON_RESPUESTA)


comparacion = pd.DataFrame([
    {
        "estrategia": nombre,
        "fragmentos": len(textos),
        "caracteres_promedio": round(np.mean([len(t) for t in textos])),
        "acierto_k1": tasa_de_acierto(textos, 1),
        "acierto_k2": tasa_de_acierto(textos, 2),
        "acierto_k4": tasa_de_acierto(textos, 4),
    }
    for nombre, textos in estrategias.items()
])
display(comparacion.style.format({"acierto_k1": "{:.0%}", "acierto_k2": "{:.0%}", "acierto_k4": "{:.0%}"}).hide(axis="index"))

vectores_elegida = np.array(embeddings.embed_documents(estrategias["Por estructura (elegida)"]))
for pregunta, esperados in PREGUNTAS_CON_RESPUESTA:
    primero = estrategias["Por estructura (elegida)"][int(np.argmax(vectores_elegida @ np.array(embeddings.embed_query(pregunta))))]
    if not all(e in primero for e in esperados):
        print(f"Fallo de la estrategia elegida con k=1: {pregunta}\n  Primer fragmento devuelto: {primero[:150]}...\n")

print("Ejemplo de un fragmento de tamaño fijo tomado del inventario:\n")
print(next(t for t in estrategias["Tamaño fijo (300)"] if "ECO-007" in t))

estrategia,fragmentos,caracteres_promedio,acierto_k1,acierto_k2,acierto_k4
Tamaño fijo (300),40,296,64%,79%,86%
Recursiva genérica (500/80),31,410,79%,86%,100%
Por estructura (elegida),44,318,93%,93%,100%


Fallo de la estrategia elegida con k=1: ¿En cuánto tiempo me devuelven la plata de una devolución?
  Primer fragmento devuelto: Política de Devoluciones y Reembolsos de EcoMarket - 2. Plazo para solicitar una devolución
El cliente dispone de 30 días calendario, contados desde l...



Ejemplo de un fragmento de tamaño fijo tomado del inventario:

go de 5 bolsas de malla para frutas y verduras, lavables",Algodón orgánico,27500,310,0,GOTS
ECO-006,Envolturas de cera de abejas,Utilidades domésticas,"Paquete de 3 envolturas reutilizables para alimentos, reemplazan el plástico de cocina",Algodón orgánico y cera de abejas,35900,75,0,GOTS
ECO-007,Bo


**Lectura del resultado.**

| Estrategia | Acierto con k=1 | Acierto con k=4 |
|---|---|---|
| Tamaño fijo | 64% (9 de 14) | 86% (12 de 14) |
| Recursiva genérica | 79% (11 de 14) | 100% |
| Por estructura | 93% (13 de 14) | 100% |

- El **tamaño fijo** es la peor estrategia y ni siquiera con cuatro fragmentos recupera todas las respuestas. El ejemplo impreso muestra la causa: el corte cae en mitad de una fila del inventario, de modo que el nombre de la botella térmica queda en un fragmento y su precio en el siguiente. Ningún fragmento contiene la respuesta completa.
- La **recursiva genérica** mejora porque corta en saltos de línea y no en mitad de una palabra, pero sigue juntando varias filas del CSV en un fragmento y mezcla productos.
- La estrategia **por estructura** deja la respuesta en el primer resultado en 13 de 14 preguntas. Su único fallo con k=1 es la pregunta por el tiempo del reembolso, donde queda primero la sección del plazo para devolver: las dos hablan de plazos de una devolución. La sección correcta aparece dentro de los cuatro primeros, y esa es la razón de usar `TOP_K = 4` y no 1.

La muestra es pequeña (cada pregunta pesa 7 puntos porcentuales), así que las diferencias entre estrategias son indicativas, no una medición definitiva.

## 8. Modelo de lenguaje

El modelo de lenguaje redacta la respuesta final a partir de los fragmentos recuperados. Se usa `ChatOpenAI` de LangChain, que habla el protocolo de la API de OpenAI. Es el mismo mecanismo del Taller 1, donde ese cliente apuntaba a Ollama en `localhost`: aquí apunta al servicio de Gemini, que expone un punto de acceso compatible. Cambiar de proveedor (Gemini, OpenAI, Ollama, Groq) solo requiere cambiar `LLM_BASE_URL`, `LLM_API_KEY` y `MODELO_LLM` en el archivo `.env`.

`temperature=0` hace que el modelo elija siempre la continuación más probable. En atención al cliente interesa que la misma pregunta reciba la misma respuesta, no la creatividad.

In [10]:
from langchain_openai import ChatOpenAI

if not LLM_API_KEY:
    raise RuntimeError("Falta LLM_API_KEY. Copia .env.example como .env y escribe ahí tu clave.")

llm = ChatOpenAI(
    model=MODELO_LLM,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0,
    max_retries=6,
    timeout=120,
)

print("Prueba de conexión:", llm.invoke("Responde únicamente con la palabra: conectado").content.strip())

Prueba de conexión: conectado


## 9. Prompt y cadena RAG

### El prompt

El prompt parte de los dos prompts del Taller 1 y conserva sus reglas (no inventar, no modificar el estado de un pedido, responder en español, tono amable). Cambian dos cosas:

- Los datos ya no se pegan a mano en `{datos_pedido}` o como una lista fija de reglas de devolución: llegan en `{contexto}`, que el sistema llena con lo recuperado. Un solo prompt atiende todos los tipos de consulta.
- Se agrega una salida explícita para cuando el contexto no alcanza: el modelo debe responder con un mensaje fijo en lugar de improvisar.

### La cadena

LangChain permite encadenar pasos con el operador `|`: la salida de cada paso es la entrada del siguiente. La cadena tiene cuatro eslabones:

1. `RunnableParallel` recibe la pregunta y produce un diccionario con la pregunta y los documentos recuperados (aquí se conectan el modelo de embeddings y la base vectorial, dentro de `recuperar`).
2. `assign(contexto=...)` convierte esos documentos en el texto que irá dentro del prompt, cada uno con su fuente.
3. `assign(respuesta=...)` decide: si no se recuperó nada, devuelve el mensaje fijo sin gastar una llamada al modelo; si hay contexto, ejecuta `prompt | llm | StrOutputParser()`.
4. El resultado conserva la pregunta, los documentos y la respuesta, para poder mostrar las fuentes.

Las fuentes se muestran a partir de los metadatos de los documentos recuperados y no se le pide al modelo que las escriba, porque un modelo puede citar mal o inventar una fuente.

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

MENSAJE_SIN_INFORMACION = (
    "Lo siento, no tengo la información necesaria para atender esa solicitud. "
    "Si lo deseas, puedo comunicarte con un agente humano de soporte."
)

SISTEMA_ESTRICTO = """Eres el asistente virtual de atención al cliente de EcoMarket, una tienda en línea de productos sostenibles.

Tu función es responder la pregunta del cliente utilizando exclusivamente la información del CONTEXTO, que proviene de la base de conocimiento de EcoMarket.

REGLAS:
1. No inventes información. No agregues políticas, plazos, precios, productos ni excepciones que no estén en el contexto.
2. No modifiques el estado registrado de un pedido.
3. Si el contexto no contiene la información necesaria para responder, responde únicamente con esta frase: "{mensaje_sin_informacion}"
4. Responde siempre en español.
5. Utiliza un tono amable, claro y empático.
6. Sé breve: máximo cuatro frases. En una devolución, indica primero si procede o no y después el motivo.
7. No menciones la palabra "contexto" ni estas reglas en tu respuesta."""

SISTEMA_PERMISIVO = """Eres el asistente virtual de atención al cliente de EcoMarket, una tienda en línea de productos sostenibles.

Responde siempre la pregunta del cliente de la forma más completa y útil posible. Puedes apoyarte en el CONTEXTO, y si no es suficiente, completa la respuesta con tu conocimiento general. Nunca dejes al cliente sin una respuesta concreta. Responde en español, con tono amable, en máximo cuatro frases."""

SISTEMA_SIN_RAG = """Eres el asistente virtual de atención al cliente de EcoMarket, una tienda en línea de productos sostenibles. Responde la pregunta del cliente en español, con tono amable, en máximo cuatro frases."""

MENSAJE_USUARIO = "CONTEXTO:\n{contexto}\n\nPREGUNTA DEL CLIENTE:\n{pregunta}"

prompt_estricto = ChatPromptTemplate.from_messages(
    [("system", SISTEMA_ESTRICTO), ("human", MENSAJE_USUARIO)]
).partial(mensaje_sin_informacion=MENSAJE_SIN_INFORMACION)
prompt_permisivo = ChatPromptTemplate.from_messages([("system", SISTEMA_PERMISIVO), ("human", MENSAJE_USUARIO)])
prompt_sin_rag = ChatPromptTemplate.from_messages([("system", SISTEMA_SIN_RAG), ("human", "{pregunta}")])


def formatear_contexto(documentos):
    return "\n\n".join(f"[Fuente: {d.metadata['fuente']}]\n{d.page_content}" for d in documentos)


def construir_cadena(prompt=prompt_estricto, k=TOP_K, umbral=UMBRAL_SIMILITUD):
    """Arma la cadena RAG. Los parámetros permiten variar el prompt y la recuperación en los experimentos."""
    generador = prompt | llm | StrOutputParser()

    def generar(entrada):
        if not entrada["documentos"]:
            return MENSAJE_SIN_INFORMACION
        return generador.invoke({"contexto": entrada["contexto"], "pregunta": entrada["pregunta"]}).strip()

    return (
        RunnableParallel(
            pregunta=RunnablePassthrough(),
            documentos=RunnableLambda(lambda pregunta: recuperar(pregunta, k=k, umbral=umbral)),
        )
        | RunnablePassthrough.assign(contexto=lambda entrada: formatear_contexto(entrada["documentos"]))
        | RunnablePassthrough.assign(respuesta=RunnableLambda(generar))
    )


cadena_rag = construir_cadena()
cadena_sin_rag = prompt_sin_rag | llm | StrOutputParser()


def preguntar(pregunta, cadena=None, titulo="RESPUESTA"):
    """Ejecuta la cadena RAG y muestra la respuesta con sus fuentes."""
    resultado = (cadena or cadena_rag).invoke(pregunta)
    print(f"\nPREGUNTA: {pregunta}")
    print(f"{titulo}: {resultado['respuesta']}")
    if resultado["documentos"]:
        fuentes = [f"{d.metadata['fuente']} ({d.metadata['similitud']:.2f})" for d in resultado["documentos"]]
        print("FUENTES:", "; ".join(fuentes))
    else:
        print("FUENTES: ninguna supera el umbral; no se llamó al modelo de lenguaje")
    time.sleep(PAUSA_ENTRE_LLAMADAS)
    return resultado


def preguntar_sin_rag(pregunta, titulo="RESPUESTA SIN RAG"):
    respuesta = cadena_sin_rag.invoke({"pregunta": pregunta}).strip()
    print(f"\nPREGUNTA: {pregunta}")
    print(f"{titulo}: {respuesta}")
    time.sleep(PAUSA_ENTRE_LLAMADAS)
    return respuesta


ejemplo = recuperar("¿Cuántos días tengo para devolver un producto?", k=2)
print("Así queda el prompt que recibe el modelo (con k=2 para que sea corto):\n")
print(prompt_estricto.format(contexto=formatear_contexto(ejemplo), pregunta="¿Cuántos días tengo para devolver un producto?"))

Así queda el prompt que recibe el modelo (con k=2 para que sea corto):

System: Eres el asistente virtual de atención al cliente de EcoMarket, una tienda en línea de productos sostenibles.

Tu función es responder la pregunta del cliente utilizando exclusivamente la información del CONTEXTO, que proviene de la base de conocimiento de EcoMarket.

REGLAS:
1. No inventes información. No agregues políticas, plazos, precios, productos ni excepciones que no estén en el contexto.
2. No modifiques el estado registrado de un pedido.
3. Si el contexto no contiene la información necesaria para responder, responde únicamente con esta frase: "Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte."
4. Responde siempre en español.
5. Utiliza un tono amable, claro y empático.
6. Sé breve: máximo cuatro frases. En una devolución, indica primero si procede o no y después el motivo.
7. No menciones la palabra "contexto" ni

## 10. Pruebas del asistente

### Los dos tipos de consulta del Taller 1

Las mismas dos consultas del Taller 1, ahora sin pegar los datos en el prompt: el sistema encuentra el pedido y las reglas de devolución por su cuenta.

In [12]:
_ = preguntar("¿Cuál es el estado de mi pedido e481f51cbdc54678b7cc49136f2d6af7?")
_ = preguntar("Hola, quiero saber cuándo llega mi pedido ee64d42b8cf066f35eac1cf57de1aa85")
_ = preguntar("Compré un perfume, ya lo abrí y lo usé, pero no me gustó. ¿Puedo devolverlo?")
_ = preguntar("El accesorio de computadora que pedí llegó defectuoso. ¿Lo puedo devolver?")


PREGUNTA: ¿Cuál es el estado de mi pedido e481f51cbdc54678b7cc49136f2d6af7?
RESPUESTA: ¡Hola! Con mucho gusto te ayudo. El estado actual de tu pedido con número e481f51cbdc54678b7cc49136f2d6af7 es "Entregado", lo cual significa que ya lo has recibido. ¡Que tengas un excelente día!
FUENTES: pedidos.csv (1.00); preguntas_frecuentes.json (0.87); preguntas_frecuentes.json (0.86); preguntas_frecuentes.json (0.85)



PREGUNTA: Hola, quiero saber cuándo llega mi pedido ee64d42b8cf066f35eac1cf57de1aa85
RESPUESTA: ¡Hola! Con mucho gusto te ayudo con tu consulta sobre tu pedido número ee64d42b8cf066f35eac1cf57de1aa85. Actualmente tu pedido se encuentra en estado "Enviado" y la fecha estimada de entrega es el 2018-06-28. Si tienes alguna otra duda, aquí estaré para ayudarte.
FUENTES: pedidos.csv (1.00); preguntas_frecuentes.json (0.89); preguntas_frecuentes.json (0.88); politica_devoluciones.pdf (0.87)



PREGUNTA: Compré un perfume, ya lo abrí y lo usé, pero no me gustó. ¿Puedo devolverlo?
RESPUESTA: Lo siento, tu devolución no procede porque los perfumes que hayan sido abiertos o utilizados no pueden ser devueltos. Si lo deseas, puedo comunicarte con un agente humano de soporte para ayudarte con cualquier otra duda.
FUENTES: politica_devoluciones.pdf (0.88); politica_devoluciones.pdf (0.86); inventario_productos.csv (0.85); politica_devoluciones.pdf (0.85)



PREGUNTA: El accesorio de computadora que pedí llegó defectuoso. ¿Lo puedo devolver?
RESPUESTA: ¡Hola! Sí puedes devolver el accesorio de computadora, ya que los productos que llegaron defectuosos pueden ser devueltos. Entiendo los inconvenientes que esto te causa y con gusto te ayudamos con el proceso.
FUENTES: politica_devoluciones.pdf (0.86); politica_devoluciones.pdf (0.86); politica_devoluciones.pdf (0.85); politica_devoluciones.pdf (0.85)


### Consultas nuevas

Preguntas que el asistente del Taller 1 no podía responder porque sus prompts no traían esa información.

In [13]:
_ = preguntar("¿Cuánto cuesta el envío y cuánto tarda en llegar a Cali?")
_ = preguntar("¿Tienen el desodorante natural? ¿Cuánto vale?")
_ = preguntar("¿Puedo pagar contra entrega?")
_ = preguntar("Quiero cancelar un pedido que ya me enviaron, ¿se puede?")


PREGUNTA: ¿Cuánto cuesta el envío y cuánto tarda en llegar a Cali?
RESPUESTA: ¡Hola! Con mucho gusto te informo que el envío estándar a Cali (ciudad principal) cuesta 9.900 pesos y es gratis en compras iguales o superiores a 150.000 pesos. Además, tu pedido tardará en llegar entre 3 y 5 días hábiles contados desde la confirmación del pago. ¡Que tengas un excelente día!
FUENTES: preguntas_frecuentes.json (0.88); preguntas_frecuentes.json (0.86); politica_devoluciones.pdf (0.84)



PREGUNTA: ¿Tienen el desodorante natural? ¿Cuánto vale?
RESPUESTA: ¡Hola! Sí tenemos el desodorante natural en crema, pero en este momento se encuentra agotado y no hay unidades disponibles. Su precio es de 21900 pesos colombianos.
FUENTES: inventario_productos.csv (0.89); inventario_productos.csv (0.85); inventario_productos.csv (0.85)



PREGUNTA: ¿Puedo pagar contra entrega?
RESPUESTA: ¡Hola! Sí, puedes pagar contra entrega en ciudades principales para compras de hasta 300.000 pesos. Además, aceptamos tarjetas de crédito y débito Visa y Mastercard, PSE y Nequi. ¡Quedamos a tu disposición para ayudarte con lo que necesites!
FUENTES: preguntas_frecuentes.json (0.86); preguntas_frecuentes.json (0.85); politica_devoluciones.pdf (0.85); preguntas_frecuentes.json (0.84)



PREGUNTA: Quiero cancelar un pedido que ya me enviaron, ¿se puede?
RESPUESTA: Lo siento, una vez que el pedido ha sido enviado ya no se puede cancelar directamente y debe tramitarse como devolución cuando llegue. ¡Con gusto te ayudaremos con ese proceso en cuanto lo recibas!
FUENTES: preguntas_frecuentes.json (0.91); preguntas_frecuentes.json (0.86); preguntas_frecuentes.json (0.86); preguntas_frecuentes.json (0.85)


## 11. Experimentos: cambios en el prompt y en la recuperación

Cada experimento cambia una sola pieza del sistema y deja el resto igual, para ver qué efecto tiene esa pieza en la respuesta.

### Experimento 2: la misma pregunta sin RAG y con RAG

Sin RAG el modelo solo tiene su conocimiento general, que no incluye las políticas ni el catálogo de EcoMarket.

In [14]:
preguntas_comparacion = [
    "¿Cuántos días tengo para devolver un producto y en cuánto tiempo me reembolsan?",
    "¿Cuánto vale el cargador solar portátil y qué garantía tiene?",
    "¿Cuál es el estado de mi pedido 1b9ecfe83cdc259250e1a8aca174f0ad?",
]
comparacion_rag = []
for pregunta in preguntas_comparacion:
    sin_rag = preguntar_sin_rag(pregunta)
    con_rag = preguntar(pregunta, titulo="RESPUESTA CON RAG")
    comparacion_rag.append({"pregunta": pregunta, "sin_rag": sin_rag, "con_rag": con_rag["respuesta"]})
    print("-" * 110)


PREGUNTA: ¿Cuántos días tengo para devolver un producto y en cuánto tiempo me reembolsan?
RESPUESTA SIN RAG: ¡Hola! En EcoMarket tienes hasta 30 días naturales a partir de la fecha de entrega para realizar cualquier devolución. Una vez que recibamos y verifiquemos el producto en nuestro almacén, el reembolso se procesará en un plazo de 5 a 7 días hábiles utilizando el mismo método de pago original. Si necesitas ayuda para iniciar el proceso, ¡estoy aquí para guiarte!



PREGUNTA: ¿Cuántos días tengo para devolver un producto y en cuánto tiempo me reembolsan?
RESPUESTA CON RAG: ¡Hola! Con gusto te ayudo con tu consulta. Tienes un plazo de 30 días calendario, contados desde la fecha en que recibiste el pedido, para solicitar una devolución. En cuanto al reembolso, el dinero se refleja entre 5 y 10 días hábiles después de que EcoMarket confirma la revisión del producto devuelto.
FUENTES: politica_devoluciones.pdf (0.92); preguntas_frecuentes.json (0.91); politica_devoluciones.pdf (0.91); politica_devoluciones.pdf (0.91)


--------------------------------------------------------------------------------------------------------------



PREGUNTA: ¿Cuánto vale el cargador solar portátil y qué garantía tiene?
RESPUESTA SIN RAG: ¡Hola! Qué gran elección para cuidar el planeta. Nuestro cargador solar portátil tiene un valor de $45 USD y cuenta con una garantía de un año por cualquier defecto de fabricación. Si necesitas ayuda para realizar tu compra o más detalles, aquí estoy para ayudarte.



PREGUNTA: ¿Cuánto vale el cargador solar portátil y qué garantía tiene?
RESPUESTA CON RAG: ¡Hola! Con mucho gusto te informo que el cargador solar portátil tiene un precio de 189900 pesos colombianos. Además, cuenta con una garantía de 12 meses para que compres con total tranquilidad. ¡Quedo a tu disposición si necesitas algo más!
FUENTES: inventario_productos.csv (0.90)


--------------------------------------------------------------------------------------------------------------



PREGUNTA: ¿Cuál es el estado de mi pedido 1b9ecfe83cdc259250e1a8aca174f0ad?
RESPUESTA SIN RAG: ¡Hola! Con gusto te ayudaré a revisar el estado de tu pedido con código 1b9ecfe83cdc259250e1a8aca174f0ad. En este momento, tus productos sostenibles ya han sido empaquetados con materiales 100% reciclados y están listos para el envío. Recibirás un correo electrónico con el número de guía y el enlace de seguimiento en las próximas 24 horas. ¡Gracias por elegirnos y por cuidar del planeta con EcoMarket!



PREGUNTA: ¿Cuál es el estado de mi pedido 1b9ecfe83cdc259250e1a8aca174f0ad?
RESPUESTA CON RAG: ¡Hola! Con mucho gusto te ayudo. El estado actual de tu pedido número 1b9ecfe83cdc259250e1a8aca174f0ad es "Cancelado". Si tienes alguna otra duda, aquí estaré para asistirte.
FUENTES: pedidos.csv (1.00); preguntas_frecuentes.json (0.88); preguntas_frecuentes.json (0.86); preguntas_frecuentes.json (0.86)


--------------------------------------------------------------------------------------------------------------


**Lectura del resultado.** Sin RAG el modelo responde las tres preguntas con total seguridad y se equivoca en las tres:

| Pregunta | Sin RAG | Con RAG | Dato real |
|---|---|---|---|
| Plazo de devolución y de reembolso | 30 días y reembolso en "5 a 7 días hábiles" | 30 días calendario y reembolso entre 5 y 10 días hábiles | 30 días; 5 a 10 días hábiles |
| Precio y garantía del cargador solar | "$45 USD" y un año de garantía | 189900 pesos colombianos y 12 meses | 189900 pesos; 12 meses |
| Estado del pedido `1b9e...` | "Empaquetado y listo para el envío", con guía en 24 horas | Cancelado | Cancelado |

- El modelo sin RAG no dice que no sabe: inventa un precio, un plazo de reembolso y el estado de un pedido. El caso del pedido es el más grave, porque le promete un envío a un cliente cuyo pedido está cancelado.
- Acertó el plazo de 30 días, pero por coincidencia: es un valor frecuente en el comercio electrónico. No hay forma de distinguir desde fuera ese acierto de sus invenciones, y eso es lo que hace inservible la respuesta sin RAG.
- Con RAG los tres datos coinciden con los documentos, y cada respuesta trae las fuentes de donde salió.

### Experimento 3: prompt estricto y prompt permisivo

Se mantiene la recuperación y se cambia solo el mensaje de sistema. Las preguntas son del dominio de EcoMarket, así que superan el umbral de similitud, pero la base de conocimiento no tiene su respuesta.

In [15]:
cadena_permisiva = construir_cadena(prompt=prompt_permisivo)

preguntas_sin_respuesta = [
    "¿Venden televisores? ¿Cuánto cuestan?",
    "¿El champú sólido sirve para cabello teñido?",
    "¿Qué descuento me dan si compro más de 10 botellas térmicas?",
]
comparacion_prompts = []
for pregunta in preguntas_sin_respuesta:
    estricta = preguntar(pregunta, titulo="PROMPT ESTRICTO")
    permisiva = preguntar(pregunta, cadena=cadena_permisiva, titulo="PROMPT PERMISIVO")
    comparacion_prompts.append({"pregunta": pregunta, "estricto": estricta["respuesta"], "permisivo": permisiva["respuesta"]})
    print("-" * 110)


PREGUNTA: ¿Venden televisores? ¿Cuánto cuestan?
PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: preguntas_frecuentes.json (0.85)



PREGUNTA: ¿Venden televisores? ¿Cuánto cuestan?
PROMPT PERMISIVO: ¡Hola! En EcoMarket nos especializamos exclusivamente en productos sostenibles y ecológicos, por lo que no vendemos electrodomésticos como los televisores. Sin embargo, en nuestro catálogo encontrarás una gran variedad de alternativas amigables con el planeta para tu hogar, cuidado personal y mucho más. ¡Echa un vistazo a nuestra web y descubre todo lo que tenemos para ti!
FUENTES: preguntas_frecuentes.json (0.85)


--------------------------------------------------------------------------------------------------------------



PREGUNTA: ¿El champú sólido sirve para cabello teñido?
PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: inventario_productos.csv (0.87)



PREGUNTA: ¿El champú sólido sirve para cabello teñido?
PROMPT PERMISIVO: El champú sólido de romero (ECO-002) está formulado específicamente para cabello graso y contiene aceites vegetales y extracto de romero. Aunque es un producto natural y libre de crueldad animal de excelente calidad, al ser clarificante para la grasa, podría acelerar un poco la pérdida del tinte en cabellos teñidos. Te recomendaría usarlo con precaución o alternarlo con un champú protector de color si deseas mantener tu tinte por más tiempo.
FUENTES: inventario_productos.csv (0.87)


--------------------------------------------------------------------------------------------------------------



PREGUNTA: ¿Qué descuento me dan si compro más de 10 botellas térmicas?
PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: inventario_productos.csv (0.87); politica_devoluciones.pdf (0.86)



PREGUNTA: ¿Qué descuento me dan si compro más de 10 botellas térmicas?
PROMPT PERMISIVO: ¡Hola! En EcoMarket contamos con la excelente botella térmica de acero (ECO-007) a un precio de 69.900 pesos colombianos, y tenemos suficientes unidades en inventario para tu pedido. Aunque en nuestro sistema actual no tenemos registrado un descuento automático por volumen al comprar más de 10 unidades, te invito a contactar directamente a nuestro equipo de atención comercial para evaluar un precio especial para ti. Recuerda que este producto incluye una garantía de 12 meses por defectos de fabricación y cuenta con certificación libre de BPA. ¡Quedamos a tu completa disposición para ayudarte con tu compra!
FUENTES: inventario_productos.csv (0.87); politica_devoluciones.pdf (0.86)


--------------------------------------------------------------------------------------------------------------


**Lectura del resultado.** Con los mismos fragmentos recuperados, el mensaje de sistema decide si el modelo se limita a lo que tiene o lo completa por su cuenta:

- **Prompt estricto:** en las tres preguntas devuelve la frase fija de "no tengo la información". Los fragmentos recuperados eran del tema (el champú, la botella térmica), pero no contenían la respuesta, y el modelo lo reconoció.
- **Prompt permisivo:** responde las tres, y en las tres agrega contenido que no está en la base de conocimiento:
  - Afirma que EcoMarket no vende televisores. Puede ser cierto, pero el modelo solo vio un fragmento de preguntas frecuentes, no el catálogo completo, así que lo está suponiendo.
  - Da un consejo sobre el cabello teñido ("podría acelerar la pérdida del tinte") que ningún documento respalda. Es una afirmación sobre un producto de cuidado personal hecha a nombre de la empresa.
  - Reconoce que no hay un descuento registrado, pero inventa un "equipo de atención comercial" que evaluaría un precio especial.

El prompt permisivo suena más servicial, y ese es el riesgo: sus respuestas son fluidas y verosímiles, y el cliente no puede saber qué parte viene de EcoMarket y qué parte inventó el modelo. Este experimento también muestra que el umbral de similitud no basta: las tres preguntas lo superaron, y lo que evitó la invención fue la regla 3 del prompt.

### Experimento 4: número de fragmentos recuperados (`k`)

Esta pregunta necesita dos partes de la política: la regla general (un perfume abierto no se devuelve) y su excepción (si llegó dañado, siempre se puede devolver). Con `k=1` el modelo solo ve un fragmento.

In [16]:
pregunta_compuesta = "Abrí un perfume y al usarlo noté que el frasco llegó roto y gotea. ¿Lo puedo devolver y quién paga el envío?"

mostrar_recuperacion(pregunta_compuesta, k=1)
mostrar_recuperacion(pregunta_compuesta, k=TOP_K)

respuesta_k1 = preguntar(pregunta_compuesta, cadena=construir_cadena(k=1), titulo="RESPUESTA CON k=1")
respuesta_k4 = preguntar(pregunta_compuesta, titulo=f"RESPUESTA CON k={TOP_K}")


Pregunta: Abrí un perfume y al usarlo noté que el frasco llegó roto y gotea. ¿Lo puedo devolver y quién paga el envío?
  0.871  politica_devoluciones.pdf  4. Productos que no se pueden devolver | Por razones sanitarias, los productos de higiene personal que hayan 

Pregunta: Abrí un perfume y al usarlo noté que el frasco llegó roto y gotea. ¿Lo puedo devolver y quién paga el envío?
  0.871  politica_devoluciones.pdf  4. Productos que no se pueden devolver | Por razones sanitarias, los productos de higiene personal que hayan 
  0.857  politica_devoluciones.pdf  5. Productos defectuosos o dañados     | Si el producto llegó defectuoso o dañado, el cliente debe reportarlo
  0.851  politica_devoluciones.pdf  3. Productos que sí se pueden devolver | Los productos que llegaron defectuosos pueden ser devueltos. Los pro
  0.851  politica_devoluciones.pdf  6. Devolución por cambio de opinión    | Cuando la devolución se debe a un cambio de opinión, el costo del en



PREGUNTA: Abrí un perfume y al usarlo noté que el frasco llegó roto y gotea. ¿Lo puedo devolver y quién paga el envío?
RESPUESTA CON k=1: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: politica_devoluciones.pdf (0.87)



PREGUNTA: Abrí un perfume y al usarlo noté que el frasco llegó roto y gotea. ¿Lo puedo devolver y quién paga el envío?
RESPUESTA CON k=4: ¡Hola! Comprendo tu situación con el perfume. Sí procede la devolución, ya que los productos que llegaron dañados por el transporte pueden ser devueltos. En este caso, EcoMarket asume el costo del envío de retorno.
FUENTES: politica_devoluciones.pdf (0.87); politica_devoluciones.pdf (0.86); politica_devoluciones.pdf (0.85); politica_devoluciones.pdf (0.85)


**Lectura del resultado.**

- **Con `k=1`** el modelo recibe solo el fragmento de la sección 4, que dice que los perfumes abiertos no se devuelven. No recibe la sección 5, que es la que trata los productos dañados y quién paga el envío. Respondió que no tiene la información. Es una respuesta prudente, pero incorrecta para el cliente: la política sí cubre su caso.
- **Con `k=4`** recibe además las secciones 5 y 3. Responde que la devolución sí procede porque el producto llegó dañado y que EcoMarket asume el envío de retorno, que es lo que dice la política.

Una pregunta real suele combinar una regla con su excepción, y esas dos partes viven en fragmentos distintos. Recuperar un solo fragmento ahorra texto en el prompt pero deja al modelo sin la mitad de la respuesta. El costo de subir `k` es un prompt más largo y más texto irrelevante; con fragmentos de unos 300 caracteres, cuatro fragmentos siguen siendo un prompt corto.

### Experimento 5: preguntas fuera de alcance y efecto del umbral

Con el umbral activo, una pregunta ajena a EcoMarket se rechaza antes de llegar al modelo. Con el umbral en 0 la pregunta llega al modelo acompañada de fragmentos que no tienen relación, y el rechazo depende solo del prompt.

In [17]:
cadena_sin_umbral = construir_cadena(umbral=0.0)
cadena_sin_umbral_permisiva = construir_cadena(prompt=prompt_permisivo, umbral=0.0)

for pregunta in ["¿Cuál es la capital de Francia?", "Dame una receta de lasaña"]:
    preguntar(pregunta, titulo="CON UMBRAL, PROMPT ESTRICTO")
    preguntar(pregunta, cadena=cadena_sin_umbral, titulo="SIN UMBRAL, PROMPT ESTRICTO")
    preguntar(pregunta, cadena=cadena_sin_umbral_permisiva, titulo="SIN UMBRAL, PROMPT PERMISIVO")
    print("-" * 110)


PREGUNTA: ¿Cuál es la capital de Francia?
CON UMBRAL, PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: ninguna supera el umbral; no se llamó al modelo de lenguaje



PREGUNTA: ¿Cuál es la capital de Francia?
SIN UMBRAL, PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: preguntas_frecuentes.json (0.74); inventario_productos.csv (0.74); politica_devoluciones.pdf (0.73); inventario_productos.csv (0.73)



PREGUNTA: ¿Cuál es la capital de Francia?
SIN UMBRAL, PROMPT PERMISIVO: La capital de Francia es París. Aunque es un dato de cultura general y no está relacionado con nuestros productos sostenibles en EcoMarket, ¡estoy aquí para ayudarte con cualquier duda que tengas! Si necesitas información sobre nuestros artículos ecológicos o el estado de tus pedidos, no dudes en consultarme.
FUENTES: preguntas_frecuentes.json (0.74); inventario_productos.csv (0.74); politica_devoluciones.pdf (0.73); inventario_productos.csv (0.73)


--------------------------------------------------------------------------------------------------------------

PREGUNTA: Dame una receta de lasaña
CON UMBRAL, PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: ninguna supera el umbral; no se llamó al modelo de lenguaje



PREGUNTA: Dame una receta de lasaña
SIN UMBRAL, PROMPT ESTRICTO: Lo siento, no tengo la información necesaria para atender esa solicitud. Si lo deseas, puedo comunicarte con un agente humano de soporte.
FUENTES: preguntas_frecuentes.json (0.82); politica_devoluciones.pdf (0.82); inventario_productos.csv (0.82); inventario_productos.csv (0.81)



PREGUNTA: Dame una receta de lasaña
SIN UMBRAL, PROMPT PERMISIVO: ¡Hola! Aunque en EcoMarket nos especializamos en productos sostenibles y no vendemos alimentos ni recetarios, con gusto te comparto una receta sencilla de lasaña vegetariana. Solo necesitas láminas de pasta, salsa de tomate natural, queso, y tus verduras sostenibles favoritas como espinacas y champiñones para el relleno. Arma capas alternando la pasta, las verduras y la salsa, cubre con queso y hornea a 180°C por unos 30 minutos hasta que gratine. ¡Buen provecho y cuéntanos si necesitas ayuda con alguno de nuestros productos ecológicos para el hogar!
FUENTES: preguntas_frecuentes.json (0.82); politica_devoluciones.pdf (0.82); inventario_productos.csv (0.82); inventario_productos.csv (0.81)


--------------------------------------------------------------------------------------------------------------


**Lectura del resultado.**

| Configuración | Capital de Francia | Receta de lasaña |
|---|---|---|
| Con umbral, prompt estricto | Rechaza sin llamar al modelo | Rechaza sin llamar al modelo |
| Sin umbral, prompt estricto | Rechaza | Rechaza |
| Sin umbral, prompt permisivo | Responde "París" | Entrega una receta |

- Con este modelo, el prompt estricto rechaza las preguntas ajenas aunque no haya umbral. El umbral no cambió la respuesta final.
- Lo que el umbral aporta es otra cosa: la pregunta se rechaza **sin llamar al modelo de lenguaje**, así que no cuesta ni tarda, y el rechazo no depende de que el modelo obedezca. Es una segunda barrera que sigue funcionando si se cambia el modelo por uno menos obediente.
- Sin umbral y con prompt permisivo, el asistente de una tienda termina dando recetas de cocina. Los fragmentos que acompañaban esas preguntas (similitud entre 0,73 y 0,82) no tenían ninguna relación, y el modelo simplemente los ignoró.

Las dos barreras cubren casos distintos: el umbral filtra las preguntas de otro tema y el prompt estricto filtra las preguntas del tema cuya respuesta no está en la base (Experimento 3).

## 12. Evaluación

Igual que en el Taller 1, se mide el sistema con un conjunto pequeño de preguntas cuya respuesta correcta se conoce. Para cada una se verifica que la respuesta contenga el dato esperado (sin distinguir mayúsculas ni tildes). En las preguntas que el sistema no puede responder, lo correcto es que diga que no tiene la información.

In [18]:
CASOS_EVALUACION = [
    # (tipo, pregunta, textos que deben aparecer en la respuesta)
    ("Pedido", "¿Cuál es el estado de mi pedido e481f51cbdc54678b7cc49136f2d6af7?", ["entregado"]),
    ("Pedido", "¿En qué va mi pedido 6942b8da583c2f9957e990d028607019?", ["enviado"]),
    ("Pedido", "Estado del pedido 714fb133a6730ab81fa1d3c1b2007291, por favor", ["cancelado"]),
    ("Pedido", "¿Qué pasó con mi pedido 15bed8e2fec7fdbadb186b57c46c92f2?", ["procesamiento"]),
    ("Devolución", "¿Cuántos días tengo para devolver un producto?", ["30"]),
    ("Devolución", "Abrí un champú y no me gustó, ¿puedo devolverlo?", ["no"]),
    ("Devolución", "El mueble llegó con daños visibles, ¿lo puedo devolver?", ["si"]),
    ("Devolución", "¿Cuánto se demora el reembolso de una devolución?", ["5", "10"]),
    ("Producto", "¿Cuánto vale la botella térmica de acero?", ["69"]),
    ("Producto", "¿Hay unidades de la lámpara de mesa de fibras naturales?", ["agotad"]),
    ("Producto", "¿De qué material es la funda de portátil?", ["corcho"]),
    ("Preguntas frecuentes", "¿Hacen envíos a otros países?", ["colombia"]),
    ("Preguntas frecuentes", "¿Desde qué valor de compra el envío es gratis?", ["150"]),
    ("Preguntas frecuentes", "¿Cuál es el horario de atención de los agentes?", ["8:00", "6:00"]),
    ("Sin respuesta", "¿Cuál es el estado de mi pedido 11111111111111111111111111111111?", ["no existe|no tengo|no encontr|no aparece|no hay"]),
    ("Sin respuesta", "¿Venden bicicletas eléctricas?", ["no tengo la informacion"]),
    ("Sin respuesta", "¿Cuál es el salario del gerente de EcoMarket?", ["no tengo la informacion"]),
    ("Sin respuesta", "¿Quién ganó el mundial de fútbol de 2014?", ["no tengo la informacion"]),
]


def normalizar(texto):
    sin_tildes = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")


def es_correcta(respuesta, esperados):
    respuesta = normalizar(respuesta)
    return all(any(re.search(rf"\b{re.escape(opcion)}", respuesta) for opcion in esperado.split("|")) for esperado in esperados)


resultados = []
for tipo, pregunta, esperados in CASOS_EVALUACION:
    salida = cadena_rag.invoke(pregunta)
    resultados.append({
        "tipo": tipo,
        "pregunta": pregunta,
        "esperado": " + ".join(esperados),
        "respuesta": salida["respuesta"],
        "fuentes": ", ".join(sorted({d.metadata["fuente"] for d in salida["documentos"]})),
        "correcta": es_correcta(salida["respuesta"], esperados),
    })
    time.sleep(PAUSA_ENTRE_LLAMADAS)

resultados_df = pd.DataFrame(resultados)
resultados_df.to_csv(RUTA_SALIDA / "resultados_evaluacion_rag.csv", index=False, encoding="utf-8-sig")

with pd.option_context("display.max_colwidth", 230):
    display(resultados_df[["tipo", "pregunta", "respuesta", "correcta"]])

resumen = resultados_df.groupby("tipo", sort=False)["correcta"].agg(correctas="sum", total="count")
display(resumen)
print(f"Respuestas correctas: {resultados_df['correcta'].sum()} de {len(resultados_df)} ({resultados_df['correcta'].mean():.0%})")
print("Resultados guardados en:", (RUTA_SALIDA / "resultados_evaluacion_rag.csv").relative_to(RUTA_BASE))

,tipo,pregunta,respuesta,correcta
0,Pedido,¿Cuál es el estado de mi pedido e481f51cbdc54678b7cc49136f2d6af7?,¡Hola! Con mucho gusto te ayudo con la información de tu compra. El estado actual de tu pedido con número e481f51cbdc54678b7cc49136f2d6af7 es Entregado. Que tengas un excelente día.,True
1,Pedido,¿En qué va mi pedido 6942b8da583c2f9957e990d028607019?,"¡Hola! Con mucho gusto te ayudo con la información de tu pedido. El estado actual de tu pedido número 6942b8da583c2f9957e990d028607019 es ""Enviado"", lo que significa que ya fue entregado a la transportadora. Su fecha estimada ...",True
2,Pedido,"Estado del pedido 714fb133a6730ab81fa1d3c1b2007291, por favor",¡Hola! Con mucho gusto te ayudo con la información de tu pedido. El estado actual de tu pedido número 714fb133a6730ab81fa1d3c1b2007291 es: Cancelado. Esto significa que fue anulado y no se despachará. ¡Que tengas un excelente ...,True
3,Pedido,¿Qué pasó con mi pedido 15bed8e2fec7fdbadb186b57c46c92f2?,"Hola, con gusto te ayudo a revisar tu pedido. Tu pedido número 15bed8e2fec7fdbadb186b57c46c92f2 se encuentra actualmente en estado ""En procesamiento"", lo que significa que el pago fue recibido y el pedido se está alistando. La...",True
4,Devolución,¿Cuántos días tengo para devolver un producto?,"¡Hola! Con gusto te informo que dispones de 30 días calendario, contados desde la fecha en que recibiste el pedido, para solicitar la devolución de un producto. Si tienes alguna otra duda, estoy aquí para ayudarte.",True
5,Devolución,"Abrí un champú y no me gustó, ¿puedo devolverlo?","Lo siento, no procede la devolución. Por razones sanitarias, los productos de higiene personal que hayan sido abiertos después de la entrega, como los champús, no pueden ser devueltos.",True
6,Devolución,"El mueble llegó con daños visibles, ¿lo puedo devolver?","¡Hola! Sí puedes devolver el mueble. Los productos que llegaron dañados por el transporte o defectuosos pueden ser devueltos, y estas exclusiones no aplican en esos casos. ¡Lamento mucho el inconveniente con tu producto!",True
7,Devolución,¿Cuánto se demora el reembolso de una devolución?,¡Hola! Con gusto te informo que el dinero del reembolso se refleja entre 5 y 10 días hábiles después de que EcoMarket confirma la revisión del producto devuelto. El dinero es enviado al mismo medio de pago que utilizaste en tu...,True
8,Producto,¿Cuánto vale la botella térmica de acero?,¡Hola! Con mucho gusto te informo que la botella térmica de acero tiene un precio de 69.900 pesos colombianos. ¡Es un producto excelente para mantener tus bebidas a la temperatura ideal! ¿Hay algo más en lo que pueda ayudarte ...,True
9,Producto,¿Hay unidades de la lámpara de mesa de fibras naturales?,"¡Hola! Lamentablemente, la lámpara de mesa de fibras naturales se encuentra actualmente agotada y no hay unidades disponibles en el inventario. Si lo deseas, puedo comunicarte con un agente humano de soporte para ayudarte con ...",True


,correctas,total
tipo,,
Pedido,4,4
Devolución,4,4
Producto,3,3
Preguntas frecuentes,3,3
Sin respuesta,4,4


Respuestas correctas: 18 de 18 (100%)
Resultados guardados en: Salida\resultados_evaluacion_rag.csv


**Lectura del resultado.** El sistema respondió correctamente las 18 preguntas: las 14 que tienen respuesta en la base de conocimiento y las 4 que no la tienen, donde lo correcto era decir que no hay información.

Ese 100% hay que leerlo con cuidado:

- **Son 18 casos escritos por el mismo equipo que construyó el sistema**, sobre una base de 54 fragmentos. Clientes reales preguntan con errores de escritura, con varias preguntas a la vez y sobre temas que nadie previó.
- **La verificación es por palabras clave.** Confirma que "30" o "cancelado" aparecen en la respuesta, no que todo lo demás que dice la respuesta sea fiel a los documentos.
- **Un caso acierta de forma mejorable.** Para el pedido `1111...`, que no existe, el contexto decía "No existe ningún pedido registrado con el número...", y el modelo respondió con la frase genérica de "no tengo la información". Cuenta como correcto porque no inventó un estado, pero habría sido más útil decirle al cliente que ese número no existe y que lo revise.

Como referencia, en el Taller 1 el modelo local respetó el estado real del pedido en el 80% de los casos. Aquí lo respetó en los 4 pedidos evaluados, pero cambiaron a la vez el modelo de lenguaje y la forma de entregarle los datos, y son solo 4 casos, así que las dos cifras no son directamente comparables.

## 13. Limitaciones y suposiciones

**Suposiciones**

- **Datos sintéticos.** EcoMarket es un caso de estudio. La política de devoluciones, el inventario y las preguntas frecuentes se redactaron para el taller. Los 10 pedidos sí son reales: son los del dataset de Olist usados en el Taller 1. Sus precios no llevan moneda porque en el origen están en reales brasileños.
- **Las reglas de devolución son las del Taller 1.** La política en PDF contiene las mismas reglas que estaban escritas en el prompt del Ejercicio 2, ampliadas con plazos y pasos.
- **Una pregunta, una respuesta.** El asistente no guarda memoria de la conversación: cada pregunta se responde de forma independiente.

**Limitaciones de recursos**

- **Equipo sin GPU y con 8 GB de RAM.** Por eso se eligió un modelo de embeddings pequeño (`multilingual-e5-small`, 384 dimensiones) en lugar de `bge-m3` o `multilingual-e5-large`, que son más precisos pero ocupan cerca de 2 GB. Tampoco se agregó una etapa de reordenamiento: el *cross-encoder* del ejemplo de clase está entrenado en inglés.
- **Modelo de lenguaje por API.** En el Taller 1 se usó Llama 3.2 en local con Ollama. Aquí se usa Gemini (`gemini-3.5-flash-lite`) por API, de modo que la pregunta del cliente y los fragmentos recuperados salen hacia un proveedor externo. Los embeddings y la base vectorial sí se ejecutan en local. El plan gratuito limita las solicitudes por minuto, y por eso hay una pausa entre llamadas.
- **Base de conocimiento pequeña.** Son 54 fragmentos. Con ese tamaño no se pone a prueba la escalabilidad de ChromaDB ni la ventaja de su índice aproximado.

**Limitaciones del diseño**

- **Los pedidos no deberían estar en el índice.** Aquí se indexan para tener un solo almacén, pero se recuperan por filtro exacto. En producción el estado de un pedido debe consultarse en la base de datos transaccional, en tiempo real y después de autenticar al cliente. En este prototipo cualquiera que conozca un número de pedido ve sus datos.
- **El inventario cambia cada minuto y el índice no.** Las existencias quedan fijas en el momento de indexar. En producción haría falta reindexar cada producto cuando cambia, o consultar las existencias directamente.
- **El umbral de relevancia es frágil.** Está calibrado con 20 preguntas y un margen de 0,046 (sección 6).
- **La evaluación es por palabras clave.** Comprueba que el dato esperado aparezca en la respuesta, no que toda la respuesta sea fiel al contexto. Son 18 casos escritos por el mismo equipo que diseñó el sistema.

## 14. Conclusiones

1. **El RAG cambió qué puede responder el asistente y qué tan confiable es.** El asistente del Taller 1 atendía dos tipos de consulta con datos pegados a mano. Con RAG, un solo prompt atiende pedidos, devoluciones, productos, envíos y pagos, y trae las fuentes de cada respuesta. Sin RAG, el mismo modelo inventó un precio, un plazo de reembolso y el estado de un pedido cancelado.

2. **Cada pieza resuelve un problema distinto, y los experimentos lo muestran por separado.**
   - El *modelo de embeddings* encuentra el fragmento aunque la pregunta use otras palabras ("me devuelven la plata" encuentra "reembolso").
   - La *segmentación* decide si la respuesta cabe en un fragmento: por estructura, el fragmento correcto quedó primero en el 93% de las preguntas; con tamaño fijo, en el 64%.
   - La *base vectorial* aporta la búsqueda por similitud y también los filtros por metadatos, que son los que permiten buscar un pedido por su número exacto.
   - El *prompt* decide qué hace el modelo cuando el contexto no alcanza: el estricto reconoce que no sabe; el permisivo inventa.
   - El *número de fragmentos* decide si llegan juntas una regla y su excepción: con `k=1` el asistente no pudo responder un caso que la política sí cubre.

3. **Decir "no sé" requirió dos barreras.** El umbral de similitud descarta las preguntas de otro tema sin gastar una llamada al modelo. La regla del prompt descarta las preguntas del tema cuya respuesta no está en la base. Ninguna de las dos basta sola.

4. **La búsqueda semántica no sirve para todo.** Para un número de pedido que no existe, devolvió el pedido de otro cliente con una similitud igual a la de un acierto. Los identificadores exactos se buscan con filtros, no con vectores.

5. **Los resultados son de un prototipo.** El 100% de la evaluación corresponde a 18 preguntas propias sobre 54 fragmentos, verificadas por palabras clave. Para llevarlo a producción haría falta evaluar con consultas reales de clientes, consultar pedidos e inventario en sus sistemas de origen y con autenticación, y medir la fidelidad de las respuestas completas.

## Anexo: cómo se generó el PDF de la política

EcoMarket es un caso de estudio, así que su política de devoluciones es un documento sintético. Esta celda contiene el texto de la política y el código que lo convierte en `Datos/politica_devoluciones.pdf`. Las reglas de las secciones 3 y 4 son las mismas del prompt de devoluciones del Taller 1, ampliadas con plazos, pasos y reembolsos.

El PDF ya está en el repositorio, por lo que la celda no lo vuelve a escribir a menos que se cambie `REGENERAR_PDF` a `True`. Si se regenera, hay que ejecutar de nuevo el notebook desde la sección 2 para reindexar.

In [19]:
REGENERAR_PDF = False
RUTA_PDF = RUTA_DATOS / "politica_devoluciones.pdf"

SECCIONES = [
    ("1. Objeto y alcance", [
        "Esta política describe las condiciones bajo las cuales los clientes de EcoMarket "
        "pueden devolver un producto y recibir un reembolso o un cambio. Aplica a todas las "
        "compras realizadas en la tienda en línea de EcoMarket con entrega dentro de Colombia.",
    ]),
    ("2. Plazo para solicitar una devolución", [
        "El cliente dispone de 30 días calendario, contados desde la fecha en que recibió el "
        "pedido, para solicitar la devolución de un producto. Las solicitudes presentadas "
        "después de ese plazo no se aceptan, salvo que apliquen las condiciones de garantía "
        "de la sección 9.",
    ]),
    ("3. Productos que sí se pueden devolver", [
        "Los productos que llegaron defectuosos pueden ser devueltos. Los productos que "
        "llegaron dañados por el transporte pueden ser devueltos. Los productos que no "
        "corresponden a lo que el cliente pidió también pueden ser devueltos.",
        "Un producto permitido puede ser devuelto aunque el cliente simplemente haya cambiado "
        "de opinión, siempre que esté sin usar, en su empaque original y con todos sus "
        "accesorios.",
    ]),
    ("4. Productos que no se pueden devolver", [
        "Por razones sanitarias, los productos de higiene personal que hayan sido abiertos "
        "después de la entrega no pueden ser devueltos. Esto incluye cepillos de dientes, "
        "champús, desodorantes y cosméticos con el sello de seguridad roto.",
        "Los perfumes que hayan sido abiertos o utilizados no pueden ser devueltos.",
        "Tampoco se aceptan devoluciones de tarjetas de regalo ni de productos "
        "personalizados a pedido del cliente.",
        "Estas exclusiones no aplican cuando el producto llegó defectuoso o dañado: en ese "
        "caso siempre puede devolverse según la sección 5.",
    ]),
    ("5. Productos defectuosos o dañados", [
        "Si el producto llegó defectuoso o dañado, el cliente debe reportarlo dentro de los "
        "5 días hábiles siguientes a la entrega, adjuntando al menos dos fotografías del "
        "producto y del empaque.",
        "En estos casos EcoMarket asume el costo del envío de retorno y el cliente puede "
        "elegir entre el reembolso total, incluido el costo de transporte original, o el "
        "reemplazo del producto sin costo adicional.",
    ]),
    ("6. Devolución por cambio de opinión", [
        "Cuando la devolución se debe a un cambio de opinión, el costo del envío de retorno "
        "corre por cuenta del cliente y el costo de transporte original no se reembolsa. "
        "Se reembolsa únicamente el precio pagado por el producto.",
    ]),
    ("7. Cómo solicitar una devolución", [
        "Paso 1: ingresar a la sección Mis pedidos de la cuenta, seleccionar el pedido y "
        "elegir la opción Solicitar devolución, o escribir a devoluciones@ecomarket.example "
        "indicando el número de pedido y el motivo.",
        "Paso 2: EcoMarket responde la solicitud en un máximo de 2 días hábiles y, si es "
        "aprobada, envía una guía de transporte al correo del cliente.",
        "Paso 3: el cliente empaca el producto y lo entrega a la transportadora dentro de "
        "los 7 días calendario siguientes a la aprobación.",
        "Paso 4: EcoMarket revisa el producto recibido en un máximo de 3 días hábiles y "
        "confirma el reembolso o el cambio.",
    ]),
    ("8. Reembolsos", [
        "El reembolso se realiza al mismo medio de pago utilizado en la compra. El dinero "
        "se refleja entre 5 y 10 días hábiles después de que EcoMarket confirma la revisión "
        "del producto devuelto. En pagos contra entrega, el reembolso se hace por "
        "transferencia a la cuenta bancaria que indique el cliente.",
        "Cuando un pedido es cancelado antes de ser enviado, el reembolso es total y se "
        "refleja entre 3 y 5 días hábiles después de la cancelación.",
    ]),
    ("9. Garantía", [
        "Los productos de la categoría Informática y accesorios y las botellas térmicas "
        "tienen garantía de 12 meses por defectos de fabricación. Los productos de la "
        "categoría Muebles y decoración tienen garantía de 6 meses. La garantía no cubre "
        "daños por mal uso, golpes o desgaste normal.",
    ]),
    ("10. Cambios por otro producto", [
        "El cliente puede pedir que el producto devuelto se cambie por otro de igual o "
        "mayor valor, pagando la diferencia. El cambio está sujeto a la disponibilidad en "
        "inventario del producto solicitado.",
    ]),
    ("11. Casos no contemplados", [
        "Las situaciones que no estén descritas en esta política son revisadas por un "
        "agente humano de soporte. El asistente virtual no está autorizado para aprobar "
        "excepciones a esta política.",
    ]),
]


def construir_html():
    partes = [
        "<h1>Política de Devoluciones y Reembolsos de EcoMarket</h1>",
        "<p>Versión 2.0. Vigente desde el 1 de enero de 2026.</p>",
    ]
    for titulo, parrafos in SECCIONES:
        partes.append(f"<h2>{titulo}</h2>")
        partes.extend(f"<p>{p}</p>" for p in parrafos)
    return "".join(partes)


def generar_pdf(ruta):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    css = (
        "body {font-family: sans-serif; font-size: 11pt; line-height: 1.35;}"
        "h1 {font-size: 17pt;} h2 {font-size: 12.5pt; margin-top: 12pt;}"
    )
    historia = pymupdf.Story(html=construir_html(), user_css=css)
    pagina = pymupdf.paper_rect("letter")
    area = pagina + (60, 60, -60, -60)
    escritor = pymupdf.DocumentWriter(str(ruta))
    quedan = True
    while quedan:
        dispositivo = escritor.begin_page(pagina)
        quedan, _ = historia.place(area)
        historia.draw(dispositivo)
        escritor.end_page()
    escritor.close()
    return ruta


if REGENERAR_PDF or not RUTA_PDF.exists():
    print("PDF generado:", generar_pdf(RUTA_PDF).name)
else:
    pdf = pymupdf.open(RUTA_PDF)
    print(f"El PDF ya existe: {RUTA_PDF.name} ({len(pdf)} páginas, {len(SECCIONES)} secciones). No se regenera.")
    pdf.close()

El PDF ya existe: politica_devoluciones.pdf (2 páginas, 11 secciones). No se regenera.
